# Observation and masks

A deterministic, synthetic example of row padding, finite-value channels, and named schemas. It uses no market data and does not measure policy behavior.

In [1]:
import numpy as np

from decisioning.observations import build_observation_schema, pad_observation_window
from simulation.environment.observations import prepare_observation_feature_channels

raw = np.array([[1.0, 2.0], [np.nan, 4.0]], dtype=np.float32)
features = prepare_observation_feature_channels(raw, include_validity=True)
window, row_mask = pad_observation_window(features, 4)
assert window.shape == (4, 4)
assert row_mask.tolist() == [0.0, 0.0, 1.0, 1.0]
assert np.isfinite(window).all()
assert window[-1, 2:].tolist() == [0.0, 1.0]
print("window shape:", window.shape, "row mask:", row_mask.tolist())

window shape: (4, 4) row mask: [0.0, 0.0, 1.0, 1.0]


In [2]:
from decisioning.trade_setup import trade_setup_observation_schema

account = build_observation_schema(entry_state_visibility=True)
print("account schema:", account)

# Mask polarity is eligibility: 1 means selectable; no-trade stays available.
setup = trade_setup_observation_schema(candidate_timeframes=("M1", "M5"))
assert setup["mask_semantics"] == "1 means eligible; NO_TRADE is always eligible"
print("setup mask schema:", setup)

account schema: ObservationSchema(version='account-state-v1', coordinates=('log_equity_over_initial', 'position_direction', 'open_pnl_over_initial', 'unrealized_r_legacy', 'initial_stop_width_over_price', 'trailing_drawdown_pct', 'entry_state_flat', 'entry_state_candidate', 'entry_state_armed', 'entry_state_in_position'), entry_state_visibility=True)
setup mask schema: {'version': 'trade-setup-observation-v2', 'candidate_schema_version': 'trade-setup-candidate-v1', 'candidate_timeframes': ['M1', 'M5'], 'completed_htf_history_enabled': True, 'eligible_candidate_ids': [1, 3, 10, 12, 19, 21, 28, 30, 37, 39, 46, 48, 55, 57, 64, 66, 73, 75, 82, 84, 91, 93, 100, 102], 'mask_key': 'trade_setup_mask', 'candidate_count': 109, 'no_trade_id': 0, 'mask_semantics': '1 means eligible; NO_TRADE is always eligible'}
